# Lesson 10 — compaction & cleanup · เก็บกวาดสิ่งที่ append ทิ้งไว้

ทุกบทที่ผ่านมาบอกว่า Lance ไม่เขียนทับ มีแต่เขียนเพิ่ม
บทนี้ดูว่าเพิ่มไปเรื่อย ๆ แล้วเกิดอะไร fragment เล็ก ๆ กองเต็ม `_deletions/` โต version พอก
แล้วสั่งเก็บกวาดสองขั้น `compact_files()` รวม fragment · `cleanup_old_versions()` ลบ version เก่า
สองคำสั่งนี้แยกกัน คนละหน้าที่ ทำทีละอย่างแล้วนับ

In [1]:
%pip install -q lancedb pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, urllib.request, pathlib, warnings
warnings.filterwarnings("ignore")   # compact_files / cleanup_old_versions warn "deprecated" — see below
if not pathlib.Path("../data/lesson_data.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Soul-Brews-Studio/lancedb-oracle/main/lessons/data/lesson_data.py", "lesson_data.py")
sys.path.insert(0, "../data")
from lesson_data import load

from pathlib import Path
import lancedb
import pandas as pd

db = lancedb.connect("./data")
tbl = db.create_table("posts", data=load("nat_posts.jsonl"), mode="overwrite")

history = []

def state(step):
    root = Path("data/posts.lance")
    row = {
        "step": step,
        "rows": tbl.count_rows(),
        "fragments": len(list((root / "data").glob("*.lance"))),
        "deletion files": len(list((root / "_deletions").glob("*"))) if (root / "_deletions").exists() else 0,
        "versions": len(list((root / "_versions").glob("*.manifest"))),
        "bytes": sum(f.stat().st_size for f in root.rglob("*") if f.is_file()),
    }
    history.append(row)
    print("  ".join(f"{k}={v}" for k, v in row.items()))

state("start")

step=start  rows=11  fragments=1  deletion files=0  versions=1  bytes=5886


[2026-09-10T11:47:26Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/10-compaction/data/posts.lance, it will be created


**กองให้ดู** — update 5 ครั้ง add 3 ครั้ง delete 2 ครั้ง รวม 10 การเขียน
ทุก `add` = fragment ใหม่ ทุก `update`/`delete` = ไฟล์ใน `_deletions/` ทุกอย่าง = version ใหม่
ตารางมีแค่ 11 แถว แต่บน disk เหมือนตารางที่ถูกเขียน 11 ครั้ง

In [3]:
for i in range(5):
    tbl.update(where=f"id = 'p0{i + 1}'", values={"topic": "memory"})
for i in range(3):
    tbl.add([{"id": f"x{i}", "date": "2026-09-10", "topic": "agents", "vector": [0.0, 1.0, 0.0], "text": f"extra {i}"}])
tbl.delete("id = 'x0'")
tbl.delete("id = 'x1'")
state("after 10 writes")

step=after 10 writes  rows=12  fragments=9  deletion files=5  versions=11  bytes=34626


**ขั้นที่ 1 — `compact_files()`** รวม fragment เล็ก ๆ เป็นก้อนเดียว
แถวที่ถูก mark ลบไว้ ถูกทิ้งจริงตอนเขียนก้อนใหม่ manifest ล่าสุดชี้ไป fragment เดียว
แต่บน disk ยังนับได้ 10 fragment 5 deletion file เพราะไฟล์เก่าไม่ถูกลบ version เพิ่มอีกสอง bytes ยิ่งโต
ตั้งแต่ 0.21 คำสั่งนี้ถูกทำเครื่องหมาย deprecated ให้ใช้ `optimize()` แทน ยังเรียกได้ เอาไว้ดูทีละขั้น

In [4]:
stats = tbl.compact_files()
print(f"compact_files: fragments_removed={stats.fragments_removed}  fragments_added={stats.fragments_added}  files_removed={stats.files_removed}  files_added={stats.files_added}")
state("after compact_files")

compact_files: fragments_removed=7  fragments_added=1  files_removed=8  files_added=1
step=after compact_files  rows=12  fragments=10  deletion files=5  versions=13  bytes=42483


**ขั้นที่ 2 — `cleanup_old_versions()`** ลบ manifest กับ fragment ที่ไม่มี version ไหนชี้ถึงแล้ว
`older_than=timedelta(0)` = ลบทุก version ที่ไม่ใช่ล่าสุด ปกติใส่ 7 วันเผื่อ reader ที่ค้างอยู่
ตรงนี้แหละที่ Nothing is Deleted หยุด เพราะเราสั่งเอง (deprecated ตั้งแต่ 0.21 เช่นกัน)
ลำดับสำคัญ compact ก่อน cleanup ทีหลัง สลับกันแล้ว cleanup ไม่มีอะไรให้ลบ
ผลคือกลับมา fragment 1 · deletion 0 · version 1 เท่าตอนเริ่ม แต่ข้อมูลเป็นชุดใหม่

In [5]:
from datetime import timedelta
stats = tbl.cleanup_old_versions(older_than=timedelta(0))
print(f"cleanup_old_versions: bytes_removed={stats.bytes_removed}  old_versions={stats.old_versions}")
state("after cleanup")

cleanup_old_versions: bytes_removed=35631  old_versions=12
step=after cleanup  rows=12  fragments=1  deletion files=0  versions=1  bytes=6852


**index ก็ค้างได้** — สร้าง FTS index แล้ว `add` แถวใหม่
ยังค้นเจอ เพราะ Lance สแกน fragment ที่ยังไม่มี index ให้แบบ brute force
แต่ `index_stats` บอกความจริง `num_unindexed_rows = 1` แถวนั้นอยู่นอก index ค้นช้ากว่า
`optimize()` = compact + cleanup + เติม index ในคำสั่งเดียว นี่คือคำสั่งที่ 0.38 อยากให้ใช้ ตั้ง schedule วันละครั้ง

In [6]:
from lancedb.index import FTS
tbl.create_index("text", config=FTS(base_tokenizer="icu"))
tbl.add([{"id": "x9", "date": "2026-09-10", "topic": "agents", "vector": [0.0, 1.0, 0.0], "text": "oracle ตัวใหม่ตื่นแล้ว"}])

def index_row(step):
    s = tbl.index_stats("text_idx")
    hits = tbl.search("oracle", query_type="fts").limit(3).to_list()
    return {"step": step, "search 'oracle' finds": " ".join(h["id"] for h in hits) or "—",
            "rows in index": s.num_indexed_rows, "rows outside index": s.num_unindexed_rows}

idx = [index_row("after add, before optimize")]
tbl.optimize(cleanup_older_than=timedelta(0))
idx.append(index_row("after optimize"))
state("after optimize")
pd.DataFrame(idx)

step=after optimize  rows=13  fragments=2  deletion files=0  versions=2  bytes=31866


,step,search 'oracle' finds,rows in index,rows outside index
0,"after add, before optimize",x9 p05 p01,12,1
1,after optimize,x9 p05 p01,13,0


**ทั้งเรื่องในตารางเดียว** — แถวละขั้น อ่านจากบนลงล่าง
`after 10 writes` fragment 9 · deletion 5 · version 11 · bytes โตเกือบ 6 เท่าจาก `start`
`after compact_files` ตัวเลขบน disk ไม่ลด กลับเพิ่ม เพราะเขียนก้อนใหม่แต่ยังไม่ทิ้งของเก่า
`after cleanup` กลับมา 1 · 0 · 1 นี่คือขั้นที่ลบจริง

ตารางที่ agent เขียนทุกวันแล้วไม่เคย optimize จะโตแบบ "after 10 writes" ไปเรื่อย ๆ
ใน fleet มีตาราง 835 MB ที่น่าจะไม่เคยผ่านคำสั่งนี้เลย
`optimize(cleanup_older_than=timedelta(days=7))` วันละครั้ง คือสิ่งที่บทนี้อยากให้จำ

In [7]:
pd.DataFrame(history)

,step,rows,fragments,deletion files,versions,bytes
0,start,11,1,0,1,5886
1,after 10 writes,12,9,5,11,34626
2,after compact_files,12,10,5,13,42483
3,after cleanup,12,1,0,1,6852
4,after optimize,13,2,0,2,31866
